In [3]:
import numpy as np
from sklearn.feature_selection import mutual_info_classif

RANDOM_STATE = 20261019

def mixed_mutual_info_scores(X, y):
    values = np.asarray(X, dtype=float)
    discrete = np.zeros(values.shape[1], dtype=bool)

    for j in range(values.shape[1]):
        col = values[:, j]
        col = col[np.isfinite(col)]
        discrete[j] = np.unique(col).size <= 2

    return mutual_info_classif(
        values,
        y,
        discrete_features=discrete,
        random_state=RANDOM_STATE
    )

In [8]:
# -*- coding: utf-8 -*-
"""
使用已保存的三个模型进行空间预测，输出 5、10、50 μg/L 的预测栅格。
"""

from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window

warnings.filterwarnings("ignore")

# --------------------------- 路径设置 ---------------------------

VARIABLE_DIR = Path(r"E:\NO-NC\normalization\variable")
REFERENCE_TIF = VARIABLE_DIR / "TPB.tif"

MODEL_ROOT = Path(
    r"E:\NO-NC\normalization\revise\reviewer3_threshold_results"
)
OUTPUT_DIR = Path(r"E:\NO-NC\normalization\revise")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLDS = (5, 10, 50)
OUTPUT_NODATA = -9999.0

# 每次预测的栅格窗口高度；可根据内存情况调大或调小
WINDOW_HEIGHT = 256


def get_model_and_features(threshold):
    """读取该阈值的最终模型及其解释变量列表。"""
    model_file = (
        MODEL_ROOT / f"threshold_{threshold}" / "final_model.joblib"
    )
    if not model_file.exists():
        raise FileNotFoundError(f"找不到模型文件：{model_file}")

    saved = joblib.load(model_file)

    if not isinstance(saved, dict) or "pipeline" not in saved:
        raise ValueError(
            f"{model_file} 中未找到 'pipeline'。"
            "请检查训练脚本保存模型时的对象结构。"
        )

    pipeline = saved["pipeline"]
    features = list(saved["features"])
    cutoff = float(saved.get("classification_threshold", np.nan))

    print(f"模型：{saved.get('model', '未记录')}")
    print(f"特征方案：{saved.get('feature_set', '未记录')}")
    print(f"管线输入变量数：{len(features)}")

    if "select" in pipeline.named_steps:
        selector = pipeline.named_steps["select"]
        print(f"SelectKBest 参数 k：{selector.k}")
        print(f"实际保留的转换后特征数：{int(selector.get_support().sum())}")

    return pipeline, features, cutoff, model_file
    


def check_raster_matches_reference(src, ref, raster_path):
    """仅检查输入栅格与 TPB.tif 的列数、行数是否一致。"""
    if src.width != ref.width or src.height != ref.height:
        raise ValueError(
            f"栅格行列数与 TPB.tif 不一致：{raster_path}\n"
            f"输入栅格：width={src.width}, height={src.height}\n"
            f"TPB参考栅格：width={ref.width}, height={ref.height}"
        )

def predict_one_threshold(threshold):
    pipeline, features, cutoff, model_file = get_model_and_features(threshold)
    
    if not REFERENCE_TIF.exists():
        raise FileNotFoundError(f"找不到参考栅格：{REFERENCE_TIF}")

    # 检查模型所需变量的栅格是否齐全
    raster_paths = {}
    missing_files = []

    for feature in features:
        tif_path = VARIABLE_DIR / f"{feature}.tif"
        if tif_path.exists():
            raster_paths[feature] = tif_path
        else:
            missing_files.append(str(tif_path))

    if missing_files:
        raise FileNotFoundError(
            "以下模型解释变量栅格未找到，请检查文件名与变量名是否一致：\n"
            + "\n".join(missing_files)
        )

    output_file = OUTPUT_DIR / f"As_risk_prediction_{threshold}ugL.tif"

    with rasterio.open(REFERENCE_TIF) as ref:
        ref_profile = ref.profile.copy()
        ref_nodata = ref.nodata

        # 输出使用 TPB.tif 的网格与空间参考信息
        output_profile = ref_profile.copy()
        output_profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=OUTPUT_NODATA,
            compress="deflate",
            predictor=3,
            BIGTIFF="IF_SAFER",
        )

        # 同时打开全部解释变量栅格
        sources = {}
        try:
            for feature, tif_path in raster_paths.items():
                src = rasterio.open(tif_path)
                check_raster_matches_reference(src, ref, tif_path)
                sources[feature] = src

            with rasterio.open(output_file, "w", **output_profile) as dst:
                for row_off in range(0, ref.height, WINDOW_HEIGHT):
                    win_height = min(WINDOW_HEIGHT, ref.height - row_off)
                    window = Window(
                        col_off=0,
                        row_off=row_off,
                        width=ref.width,
                        height=win_height,
                    )

                    # 参考栅格有效区域定义输出有效区域
                    ref_array = ref.read(1, window=window)
                    if ref_nodata is None:
                        ref_valid = np.isfinite(ref_array)
                    else:
                        ref_valid = (
                            np.isfinite(ref_array)
                            & (ref_array != ref_nodata)
                        )

                    n_pixels = win_height * ref.width
                    X_block = {}

                    for feature in features:
                        src = sources[feature]
                        arr = src.read(1, window=window).astype(
                            np.float32, copy=False
                        )

                        # 栅格 NoData 及非有限值转成 NaN，
                        # 交由保存的模型预处理管线按训练规则填补
                        if src.nodata is not None:
                            arr = np.where(
                                arr == src.nodata, np.nan, arr
                            )
                        arr = np.where(np.isfinite(arr), arr, np.nan)

                        X_block[feature] = arr.reshape(-1)

                    X_df = pd.DataFrame(X_block, columns=features)

                    # 仅对 TPB 有效空间范围内的像元预测
                    valid_flat = ref_valid.reshape(-1)
                    prediction = np.full(
                        n_pixels, OUTPUT_NODATA, dtype=np.float32
                    )

                    if valid_flat.any():
                        scores = np.asarray(
                            pipeline.predict(X_df.loc[valid_flat]),
                            dtype=np.float32
                        ).reshape(-1)

                        prediction[valid_flat] = scores

                    dst.write(
                        prediction.reshape(win_height, ref.width),
                        1,
                        window=window,
                    )

        finally:
            for src in sources.values():
                src.close()

    print(f"阈值 {threshold} μg/L")
    print(f"  模型：{model_file}")
    print(f"  变量数：{len(features)}")
    print(f"  分类界值：{cutoff}")
    print(f"  输出：{output_file}")


def main():
    for threshold in THRESHOLDS:
        predict_one_threshold(threshold)

    print(f"\n三个阈值的空间预测栅格已保存至：{OUTPUT_DIR}")


if __name__ == "__main__":
    main()

模型：RandomForestRegressor
特征方案：all
管线输入变量数：31
阈值 5 μg/L
  模型：E:\NO-NC\normalization\revise\reviewer3_threshold_results\threshold_5\final_model.joblib
  变量数：31
  分类界值：0.2
  输出：E:\NO-NC\normalization\revise\As_risk_prediction_5ugL.tif
模型：XGBoostRegressor
特征方案：SelectKBest
管线输入变量数：31
SelectKBest 参数 k：20
实际保留的转换后特征数：20
阈值 10 μg/L
  模型：E:\NO-NC\normalization\revise\reviewer3_threshold_results\threshold_10\final_model.joblib
  变量数：31
  分类界值：0.15
  输出：E:\NO-NC\normalization\revise\As_risk_prediction_10ugL.tif
模型：LightGBMRegressor
特征方案：correlation
管线输入变量数：28
阈值 50 μg/L
  模型：E:\NO-NC\normalization\revise\reviewer3_threshold_results\threshold_50\final_model.joblib
  变量数：28
  分类界值：0.2
  输出：E:\NO-NC\normalization\revise\As_risk_prediction_50ugL.tif

三个阈值的空间预测栅格已保存至：E:\NO-NC\normalization\revise
